# 作业四：机器翻译计算BLEU
姓名：
学号：

## 加载模型和Tokenizer

本次作业将使用从huggingface中下载德英翻译模型`Helsinki-NLP/opus-mt-en-de`进行翻译测试，无需大家自行训练模型

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")


### Tokenizer

> 下面是一个例子，展示Tokenizer和模型的使用。理解下面的例子可能对你的作业有帮助。

```python

Tokenizer会将句子分割成一个个token，然后将每个token转化为一个数字，这个数字就是这个token在词表中的id。

In [ ]:
# Sample English sentence
sentence = "Good morning! How are you?"
inputs = tokenizer.encode(sentence, return_tensors="pt")
print(inputs)

可以将token id映射到对应的分词token

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs[0])
print(tokens)

可以使用`decode`方法将token id转化回原来的句子

In [ ]:
decoded_string = tokenizer.decode(inputs[0])
print(decoded_string)

## 计算BLEU score

BLEU 是机器翻译评估中最常用的指标之一。

核心思想：

- 一个好的翻译结果，应该在 n-gram 上尽可能和参考翻译重叠。
- 为了避免模型生成过短的翻译，BLEU 引入了 简短惩罚（Brevity Penalty, BP）。

计算公式为：
$$
BLEU = BP \cdot \exp(\sum^N_{n=1} w_n \log p_n)
$$
其中：

- $p_n$：n-gram 的修正精确率（modified precision）
- $w_n$：权重（通常取均匀分布，$w_n = 1/N$）
- BP：简短惩罚，定义如下
$$
B P= \begin{cases}1 & \text { if } c>r \\ \exp \left(1-\frac{r}{c}\right) & \text { if } c \leq r\end{cases}
$$

- $c$：候选翻译的长度
- $r$：参考翻译的长度（取最接近候选翻译长度的参考）

### n-gram提取

In [ ]:
from collections import Counter

def ngram_counts(tokens, n):
    """
    从一个分词后的句子中提取所有 n-gram，并统计频数。
    
    参数:
        tokens (list): 句子的分词结果
        n (int): n-gram 的阶数
    
    返回:
        Counter: n-gram -> 出现次数
    """
    ## TODO：统计句子中 n-gram 的频次（2分）


### 修正精确率（modified precision）

In [ ]:
def modified_precision(candidate, references, n):
    """
    计算 n-gram 的修正精确率。
    
    参数:
        candidate (list): 候选翻译（分词后的列表）
        references (list of list): 多个参考翻译（每个参考为一个分词列表）
        n (int): n-gram 阶数
    
    返回:
        (clipped_count, total_count): clip 后的匹配数量与候选总数
    """
    cand_ngrams = ngram_counts(candidate, n)
    max_ref_counts = Counter()

    # TODO：计算所有参考翻译中的最大 n-gram 计数（2分）

    # 对候选中的 n-gram 进行截断计数
    clipped = {ng: min(count, max_ref_counts[ng]) for ng, count in cand_ngrams.items()}
    
    # TODO：返回匹配与候选的总数（1分）


### 简短惩罚（Brevity Penalty）

In [ ]:
import math

def brevity_penalty(candidate, references):
    """
    计算简短惩罚 (BP)。
    
    参数:
        candidate (list): 候选翻译
        references (list of list): 参考翻译列表
    
    返回:
        float: BP 值
    """
    c = len(candidate)
    ref_lens = [len(ref) for ref in references]
    # TODO：选择与候选长度最接近的参考长度并计算BP值（3分）


### BLEU分数

In [ ]:
def bleu_score(candidate, references, max_n=4):
    """
    计算 BLEU 分数（默认 BLEU-4）。
    
    参数:
        candidate (list): 候选翻译
        references (list of list): 参考翻译
        max_n (int): 最大 n-gram 阶数（默认为 4）
    
    返回:
        float: BLEU 分数
    """
    # TODO：根据公式和以上函数计算BLEU分数（6分）


## 测试与总结（6分）

### Sample 1

In [ ]:
sentence = "The weather is nice today."
references = [
    ["Das", "Wetter", "ist", "heute", "schön", "."],
    ["Heute", "ist", "das", "Wetter", "schön", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=40, num_beams=4, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence)

print("Sample-1 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-1 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-1 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


### Sample 2

In [ ]:
sentence = "The conference will take place in Berlin next week, and many researchers from different countries are expected to attend."
references = [
    ["Die", "Konferenz", "findet", "nächste", "Woche", "in", "Berlin", "statt", ",", "und", "es", "wird", "erwartet", ",", "dass", "viele", "Forscher", "aus", "verschiedenen", "Ländern", "teilnehmen", "."],
    ["Nächste", "Woche", "wird", "die", "Konferenz", "in", "Berlin", "abgehalten", ",", "wobei", "zahlreiche", "Wissenschaftler", "aus", "unterschiedlichen", "Ländern", "erwartet", "werden", "."],
    ["Die", "Konferenz", "wird", "nächste", "Woche", "in", "Berlin", "stattfinden", ",", "und", "viele", "Forscher", "aus", "anderen", "Ländern", "sollen", "daran", "teilnehmen", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=64, num_beams=4, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence.split())

print("Sample-2 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-2 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-2 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


### Sample 3

In [ ]:
sentence = "Artificial intelligence is rapidly developing, and it is increasingly being applied in areas such as healthcare, education, and autonomous driving."
references = [
    ["Die", "künstliche", "Intelligenz", "entwickelt", "sich", "rasant", "und", "wird", "zunehmend", "in", "Bereichen", "wie", "Gesundheitswesen", ",", "Bildung", "und", "autonomes", "Fahren", "eingesetzt", "."],
    ["Künstliche", "Intelligenz", "macht", "schnelle", "Fortschritte", "und", "findet", "immer", "häufiger", "Anwendung", "im", "Gesundheitswesen", ",", "in", "der", "Bildung", "und", "im", "autonomen", "Fahren", "."],
    ["Künstliche", "Intelligenz", "entwickelt", "sich", "schnell", "und", "wird", "mehr", "in", "Medizin", ",", "Schulen", "und", "selbstfahrenden", "Autos", "benutzt", "."]
]

# Translate English to Deutsch (German)
inputs = tokenizer.encode(sentence, return_tensors="pt")
outputs = model.generate(inputs, max_length=40, num_beams=5, early_stopping=True)
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Translated Sentence:", translated_sentence)

print("Sample-3 BLEU-1:", bleu_score(translated_sentence.split(), references, max_n=1))
print("Sample-3 BLEU-2:", bleu_score(translated_sentence.split(), references, max_n=2))
print("Sample-3 BLEU-4:", bleu_score(translated_sentence.split(), references, max_n=4))


（TODO：实验总结）